# 13. Benchmark Genre and Decade Classification

**Question:** Do transposition-invariant harmonic trigrams generalize better than literal chord trigrams?

**Deliverable:** An artist-held-out benchmark with interpretable errors and feature associations.

```mermaid
flowchart LR
    A["Song chord sequences"] --> B["Literal and H3 features"]
    B --> C["Artist-held-out models"]
    C --> D["Metrics and confusion"]
    D --> E["Interpretable findings"]
```

Both representations use the same songs, split, weighting, and linear classifier. The representation is the controlled difference.


## Benchmark design

Compare a most-frequent baseline, literal chord trigrams, and transposition-invariant H3 features. Songs by the same artist never cross the train-test boundary.

The notebook caps songs per class for laptop-friendly runtime. It is a representation benchmark, not a hyperparameter search.


## Setup

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfTransformer, TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import GroupShuffleSplit

CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
SONGS_PATH = ROOT / "data" / "processed" / "songs_master.parquet"
DB_PATH = ROOT / "data" / "processed" / "harmonic_trends.duckdb"

print({"root": str(ROOT), "songs": str(SONGS_PATH), "database": str(DB_PATH)})

## Parameters

`MAX_SONGS_PER_CLASS` controls runtime. Raising it uses more data. `MAX_FEATURES` limits each representation to its most common trigrams.

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.20
MAX_SONGS_PER_CLASS = 3_000
MAX_FEATURES = 5_000
MIN_EXACT_DOCUMENT_FREQUENCY = 5
HARMONIC_N = 3

## Load the working data

The metadata is small enough to load once. Chord sequences are attached only after sampling, which avoids bringing every full progression into memory.

In [ ]:
def sample_per_class(frame, target):
    samples = []
    for _, group in frame.groupby(target):
        n = min(len(group), MAX_SONGS_PER_CLASS)
        samples.append(group.sample(n=n, random_state=RANDOM_STATE))
    return pd.concat(samples, ignore_index=True)


def attach_chord_sequences(frame):
    with duckdb.connect() as con:
        con.register("selected_songs", frame[["song_id"]])
        sequences = con.execute(
            """
            SELECT
                p.id::BIGINT AS song_id,
                p.chords_nosections
            FROM read_parquet(?) AS p
            JOIN selected_songs AS s ON p.id = s.song_id
            """,
            [str(SONGS_PATH)],
        ).fetchdf()

    return (
        frame.merge(sequences, on="song_id")
        .dropna(subset=["chords_nosections"])
        .sort_values("song_id")
        .reset_index(drop=True)
    )


def prepare_task(metadata, target):
    frame = metadata.dropna(subset=[target, "artist_id"]).copy()
    if target == "decade":
        frame = frame[frame["decade"] >= 1950]
        frame["decade"] = frame["decade"].astype(int)

    frame = sample_per_class(frame, target)
    return attach_chord_sequences(frame)


with duckdb.connect() as con:
    metadata = con.execute(
        """
        SELECT
            id::BIGINT AS song_id,
            artist_id,
            main_genre,
            CAST(decade AS INTEGER) AS decade_value
        FROM read_parquet(?)
        WHERE artist_id IS NOT NULL
        """,
        [str(SONGS_PATH)],
    ).fetchdf()

metadata = metadata.rename(columns={"decade_value": "decade"})
genre_songs = prepare_task(metadata, "main_genre")
decade_songs = prepare_task(metadata, "decade")

print({"genre_songs": len(genre_songs), "decade_songs": len(decade_songs)})

In [ ]:
display(
    genre_songs["main_genre"]
    .value_counts()
    .sort_index()
    .rename("songs")
    .to_frame()
)

display(
    decade_songs["decade"]
    .value_counts()
    .sort_index()
    .rename("songs")
    .to_frame()
)

## Benchmark helpers

The literal representation is built directly from chord text. The harmonic representation uses the song-level `H3` counts created in notebook 12.

TF-IDF is fitted only on the training rows. The classifier uses logistic loss and stochastic gradient descent, which is a fast way to fit a linear classifier to sparse text-like features.

In [ ]:
def artist_split(frame, target):
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
    train_index, test_index = next(
        splitter.split(frame, frame[target], groups=frame["artist_id"])
    )

    train_artists = set(frame.iloc[train_index]["artist_id"])
    test_artists = set(frame.iloc[test_index]["artist_id"])
    print(
        {
            "train_songs": len(train_index),
            "test_songs": len(test_index),
            "artist_overlap": len(train_artists & test_artists),
        }
    )
    return train_index, test_index


def literal_trigram_features(frame, train_index, test_index):
    vectorizer = TfidfVectorizer(
        tokenizer=str.split,
        token_pattern=None,
        lowercase=False,
        ngram_range=(3, 3),
        min_df=MIN_EXACT_DOCUMENT_FREQUENCY,
        max_features=MAX_FEATURES,
        sublinear_tf=True,
    )
    train_matrix = vectorizer.fit_transform(
        frame.iloc[train_index]["chords_nosections"]
    )
    test_matrix = vectorizer.transform(
        frame.iloc[test_index]["chords_nosections"]
    )
    return train_matrix, test_matrix, vectorizer


def harmonic_count_matrix(frame, train_index):
    with duckdb.connect(str(DB_PATH), read_only=True) as con:
        con.register("training_songs", frame.iloc[train_index][["song_id"]])
        features = con.execute(
            """
            SELECT
                t.harmonic_id,
                ANY_VALUE(d.example_ngram) AS example_ngram,
                COUNT(*)::BIGINT AS song_df
            FROM song_harmonic_terms AS t
            JOIN training_songs AS s USING (song_id)
            JOIN harmonic_song_document_frequency AS d
                ON t.n = d.n AND t.harmonic_id = d.harmonic_id
            WHERE t.n = ?
            GROUP BY t.harmonic_id
            ORDER BY song_df DESC
            LIMIT ?
            """,
            [HARMONIC_N, MAX_FEATURES],
        ).fetchdf()

        con.register("selected_songs", frame[["song_id"]])
        con.register("selected_features", features[["harmonic_id"]])
        terms = con.execute(
            """
            SELECT t.song_id, t.harmonic_id, t.count
            FROM song_harmonic_terms AS t
            JOIN selected_songs AS s USING (song_id)
            JOIN selected_features AS f USING (harmonic_id)
            WHERE t.n = ?
            """,
            [HARMONIC_N],
        ).fetchdf()

    row_lookup = pd.Series(np.arange(len(frame)), index=frame["song_id"])
    column_lookup = pd.Series(np.arange(len(features)), index=features["harmonic_id"])

    matrix = sparse.csr_matrix(
        (
            terms["count"].astype(float),
            (
                terms["song_id"].map(row_lookup),
                terms["harmonic_id"].map(column_lookup),
            ),
        ),
        shape=(len(frame), len(features)),
    )
    return matrix, features

In [ ]:
def make_classifier():
    return SGDClassifier(
        loss="log_loss",
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )


def top_k_accuracy(model, matrix, truth, k=3):
    probabilities = model.predict_proba(matrix)
    top_columns = np.argpartition(probabilities, -k, axis=1)[:, -k:]
    return np.mean(
        [label in model.classes_[columns] for label, columns in zip(truth, top_columns)]
    )


def evaluate_model(name, model, train_matrix, test_matrix, y_train, y_test):
    model.fit(train_matrix, y_train)
    predictions = model.predict(test_matrix)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
        "macro_f1": f1_score(y_test, predictions, average="macro"),
        "top3_accuracy": top_k_accuracy(model, test_matrix, y_test, k=3),
    }

    if np.issubdtype(np.asarray(y_test).dtype, np.number):
        metrics["mean_absolute_decades"] = np.mean(
            np.abs(np.asarray(y_test) - predictions)
        ) / 10

    report = classification_report(
        y_test,
        predictions,
        output_dict=True,
        zero_division=0,
    )
    return metrics, model, predictions, report


def run_benchmark(frame, target):
    train_index, test_index = artist_split(frame, target)
    y = frame[target].to_numpy()
    y_train, y_test = y[train_index], y[test_index]

    literal_train, literal_test, literal_vectorizer = literal_trigram_features(
        frame, train_index, test_index
    )

    harmonic_counts, harmonic_features = harmonic_count_matrix(frame, train_index)
    harmonic_tfidf = TfidfTransformer(sublinear_tf=True)
    harmonic_train = harmonic_tfidf.fit_transform(harmonic_counts[train_index])
    harmonic_test = harmonic_tfidf.transform(harmonic_counts[test_index])

    print(
        {
            "literal_shape": literal_train.shape,
            "harmonic_shape": harmonic_train.shape,
        }
    )

    specifications = [
        (
            "Most frequent",
            DummyClassifier(strategy="most_frequent"),
            harmonic_train,
            harmonic_test,
        ),
        (
            "Literal chord trigrams",
            make_classifier(),
            literal_train,
            literal_test,
        ),
        (
            "Harmonic trigrams (H3)",
            make_classifier(),
            harmonic_train,
            harmonic_test,
        ),
    ]

    rows = []
    models = {}
    predictions = {}
    reports = {}

    for name, model, train_matrix, test_matrix in specifications:
        metrics, fitted_model, model_predictions, report = evaluate_model(
            name,
            model,
            train_matrix,
            test_matrix,
            y_train,
            y_test,
        )
        rows.append(metrics)
        models[name] = fitted_model
        predictions[name] = model_predictions
        reports[name] = report

    results = pd.DataFrame(rows).dropna(axis=1, how="all")
    artifacts = {
        "frame": frame,
        "target": target,
        "train_index": train_index,
        "test_index": test_index,
        "y_test": y_test,
        "models": models,
        "predictions": predictions,
        "reports": reports,
        "literal_vectorizer": literal_vectorizer,
        "harmonic_features": harmonic_features,
    }
    return results, artifacts

In [ ]:
def plot_model_comparison(results, title):
    columns = ["accuracy", "balanced_accuracy", "macro_f1", "top3_accuracy"]
    ax = results.set_index("model")[columns].plot.bar(figsize=(10, 4))
    ax.set_title(title)
    ax.set_ylabel("score")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=15)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
    plt.tight_layout()
    plt.show()


def show_confusion(artifacts, title):
    model_name = "Harmonic trigrams (H3)"
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay.from_predictions(
        artifacts["y_test"],
        artifacts["predictions"][model_name],
        normalize="true",
        values_format=".2f",
        cmap="Blues",
        xticks_rotation=45,
        ax=ax,
    )
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def top_harmonic_features(artifacts, features_per_class=4):
    model = artifacts["models"]["Harmonic trigrams (H3)"]
    features = artifacts["harmonic_features"].reset_index(drop=True)
    rows = []

    for class_index, label in enumerate(model.classes_):
        strongest = np.argsort(model.coef_[class_index])[-features_per_class:][::-1]
        for rank, feature_index in enumerate(strongest, start=1):
            feature = features.iloc[feature_index]
            rows.append(
                {
                    "label": label,
                    "rank": rank,
                    "example_progression": feature["example_ngram"],
                    "weight": model.coef_[class_index, feature_index],
                }
            )

    return pd.DataFrame(rows)


def explain_findings(results, artifacts, task_name):
    result_lookup = results.set_index("model")
    literal_f1 = result_lookup.loc["Literal chord trigrams", "macro_f1"]
    harmonic_f1 = result_lookup.loc["Harmonic trigrams (H3)", "macro_f1"]
    difference = harmonic_f1 - literal_f1

    report = artifacts["reports"]["Harmonic trigrams (H3)"]
    labels = artifacts["models"]["Harmonic trigrams (H3)"].classes_
    recalls = {str(label): report[str(label)]["recall"] for label in labels}
    easiest = max(recalls, key=recalls.get)
    hardest = min(recalls, key=recalls.get)

    if difference > 0:
        comparison = "outperformed"
    else:
        comparison = "did not outperform"

    print(
        f"For {task_name}, harmonic trigrams {comparison} literal chord trigrams "
        f"by {abs(difference):.3f} macro-F1."
    )
    print(
        f"The harmonic model recalled {easiest} most reliably "
        f"({recalls[easiest]:.1%}) and {hardest} least reliably "
        f"({recalls[hardest]:.1%})."
    )

## 1. Genre classification

The genre task uses the dataset's 12 broad `main_genre` labels. Because these labels are attached at the artist level, the artist-held-out split is essential.

In [ ]:
genre_results, genre_artifacts = run_benchmark(genre_songs, "main_genre")
display(genre_results.round(3))
plot_model_comparison(genre_results, "Genre classification")

In [ ]:
show_confusion(
    genre_artifacts,
    "Genre confusion matrix: harmonic trigrams",
)

The confusion matrix is normalized by the true genre. Read each row as: “Of the songs that actually belong to this genre, where did the model place them?” Similar genres may remain difficult to separate because harmony is only one part of musical style.

In [ ]:
explain_findings(genre_results, genre_artifacts, "genre classification")
display(top_harmonic_features(genre_artifacts, features_per_class=3))

## 2. Decade classification

The decade task uses songs from the 1950s onward. In addition to classification scores, we report the mean absolute error in decades. Predicting a neighboring decade is less severe than missing by several decades.

In [ ]:
decade_results, decade_artifacts = run_benchmark(decade_songs, "decade")
display(decade_results.round(3))
plot_model_comparison(decade_results, "Decade classification")

In [ ]:
show_confusion(
    decade_artifacts,
    "Decade confusion matrix: harmonic trigrams",
)

In [ ]:
explain_findings(decade_results, decade_artifacts, "decade classification")
display(top_harmonic_features(decade_artifacts, features_per_class=3))

## Findings

In [ ]:
summary = pd.concat(
    [
        genre_results.assign(task="genre"),
        decade_results.assign(task="decade"),
    ],
    ignore_index=True,
)
display(summary.set_index(["task", "model"]).round(3))

for task, results in [("genre", genre_results), ("decade", decade_results)]:
    scores = results.set_index("model")["macro_f1"]
    difference = scores["Harmonic trigrams (H3)"] - scores["Literal chord trigrams"]
    direction = "supports" if difference > 0 else "does not support"
    print(
        f"The {task} result {direction} the hypothesis that transposition-invariant "
        f"trigrams generalize better. Macro-F1 difference: {difference:+.3f}."
    )

### Default-run result

With the parameters above, the artist-held-out test contains **7,166 songs for genre** and **4,595 songs for decade**, with zero artist overlap.

- Genre macro-F1 rises from **0.185** with literal trigrams to **0.205** with harmonic trigrams. Top-3 accuracy rises from **43.3%** to **46.4%**.
- Decade macro-F1 rises from **0.205** to **0.219**. Mean absolute error falls from **1.96 decades** to **1.85 decades**.

The improvement is modest but consistent across both tasks. This supports the representation hypothesis: grouping transposed versions of the same progression helps the model generalize to unseen artists. The absolute scores also show that harmony alone does not fully determine genre or era, which is an important finding rather than a failure.

### Interpretation

A harmonic-model gain supports the value of transposition invariance; a loss suggests removed information or insufficient context. Feature weights are associations, not causes.

The published Chordonomicon accuracies are context only because this notebook uses a stricter sampled, artist-held-out split.


## Limitation and next experiment

This first pass uses trigrams and one linear classifier. Next, compare H3 through H8 on the same fixed split before adding model complexity.
